In [46]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import os 
from padelpy import padeldescriptor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from rdkit.Chem import rdFingerprintGenerator  # Import the modern generator module


import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. LOAD AND CLEAN THE DATASET
# ==========================================
# Load the exported CSV file
df = pd.read_excel("FINAL.xlsx")
df


,COUNT,cid,SMILES,MW_pub,ck pub,VERIFY,Unnamed: 6,ck_swis,MW_swis,ESOL Class,...,Unnamed: 12,CK_AD,MW_AD,caco2,hia,BBB,hERG,hERG-10um,Ames,H-HT
0,1,145991278.0,C1[C@]2(C3=CC(=NC=C3OC4=C2C=C(C=C4)C5=NC(=CC=C...,443.4,443,True,NaN,443.0,443.41,Moderately soluble,...,NaN,443.0,443.12,-4.881416,0.000000,0.001933,0.909164,0.717378,0.947849,0.989082
1,2,145990468.0,CC#CC1=CC(=CN=C1)C2=CC3=C(C=C2)OC4=CN=C(C=C4[C...,463.5,463,True,NaN,463.0,463.46,Moderately soluble,...,NaN,463.0,463.14,-4.828875,0.000000,0.000030,0.942825,0.776856,0.794406,0.988022
2,3,122172743.0,CN1C(NC2(C1=O)C3=C(C=CC(=C3)OCC(F)(F)F)OC24CCC...,447.4,447,True,NaN,447.0,447.45,Moderately soluble,...,NaN,447.0,447.18,-5.399611,0.000046,0.969402,0.755374,0.757735,0.426495,0.804849
3,4,145983198.0,CC#CC1=CC(=CN=C1)C2=CC3=C(C=C2)OC4=CN=C(C=C4[C...,450.5,450,True,NaN,450.0,450.49,Moderately soluble,...,NaN,450.0,450.17,-4.864286,0.000006,0.004028,0.885521,0.696834,0.734198,0.973429
4,5,45254510.0,CC1(COC1)C#CC2=CC3=C(N=C2)OC4=C([C@@]35COC(=N5...,442.4,442,True,NaN,442.0,442.44,Moderately soluble,...,NaN,442.0,442.14,-4.885711,0.000002,0.698582,0.770069,0.598314,0.942510,0.976820
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3514,3515,NaN,NaN,NaN,0,True,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3515,3516,NaN,NaN,NaN,0,True,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3516,3517,NaN,NaN,NaN,0,True,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3517,3518,NaN,NaN,NaN,0,True,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df.columns = df.columns.str.strip()

target_endpoint = "hERG"
df = df.dropna(subset=["SMILES", target_endpoint])

# ==========================================
# 2. MODERN FINGERPRINT GENERATION
# ==========================================
# Initialize the modern Morgan Fingerprint Generator
# radius=2 is equivalent to ECFP4
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


def smiles_to_morgan_generator_fp(smiles):
    """Converts a SMILES string into a NumPy array using the modern MorganGenerator."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Use the generator to get a NumPy bit vector directly
    fp = morgan_gen.GetCountFingerprintAsNumPy(mol)
    return fp


print("Generating fingerprints using the new MorganGenerator API...")
df["Fingerprint"] = df["SMILES"].apply(smiles_to_morgan_generator_fp)
df = df.dropna(subset=["Fingerprint"])

X = np.array(df["Fingerprint"].tolist())
y = df[target_endpoint].values
print(f"Features matrix shape: {X.shape}")

# ==========================================
# 3. TRAINING AND EVALUATION
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("\n=== MODEL PERFORMANCE METRICS ===")
print(f"Mean Squared Error (MSE): {mean_squared_error(y_test, y_pred):.4f}")
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")

# Mean Squared Error (MSE): 0.0029
# Coefficient of Determination (R² Score): 0.9433

Generating fingerprints using the new MorganGenerator API...
Features matrix shape: (2856, 2048)

=== MODEL PERFORMANCE METRICS ===
Mean Squared Error (MSE): 0.0027
R² Score: 0.9479


In [43]:

df.columns = df.columns.str.strip()

# Choose your target toxicity column (e.g., 'hERG', 'Ames', 'H-HT', or 'BBB')
target_endpoint = "hERG"

# Remove rows containing missing values in SMILES or the target column
df = df.dropna(subset=["SMILES", target_endpoint])


# ==========================================
# 2. MOLECULAR FINGERPRINT EXTRACTION (ECFP4)
# ==========================================
def smiles_to_morgan_fp(smiles, radius=2, n_bits=2048):
    """Converts a SMILES string into a dense NumPy array representing a Morgan Fingerprint."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  # Return None if the SMILES string is invalid/cannot be parsed

    # Generate Morgan fingerprint as a bit vector
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)

    # Convert the RDKit bit vector to a dense numpy array
    arr = np.zeros((1,))
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


print("Generating chemical fingerprints from SMILES strings...")
df["Fingerprint"] = df["SMILES"].apply(smiles_to_morgan_fp)

# Drop any rows where the SMILES string failed to convert properly
df = df.dropna(subset=["Fingerprint"])

# Construct the feature matrix X and target vector y
X = np.array(df["Fingerprint"].tolist())
y = df[target_endpoint].values

print(f"Dataset successfully featurized. Matrix shape: {X.shape}")


# ==========================================
# 3. DATA SPLITTING & MACHINE LEARNING
# ==========================================
# Split into 80% training data and 20% test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]} molecules")
print(f"Testing set size: {X_test.shape[0]} molecules")

# Initialize and train the Random Forest Regressor
print(f"Training Random Forest model to predict {target_endpoint}...")
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)


# ==========================================
# 4. MODEL EVALUATION
# ==========================================
# Predict on the unseen test set
y_pred = model.predict(X_test)

# Calculate metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n=== MODEL PERFORMANCE METRICS ===")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Coefficient of Determination (R² Score): {r2:.4f}")


# ==========================================
# 5. PREDICTING ON NEW, UNKNOWN MOLECULES
# ==========================================
def predict_new_molecule(new_smiles_string):
    """Predicts the toxicity score for a brand-new molecule string."""
    fp = smiles_to_morgan_fp(new_smiles_string)
    if fp is None:
        return "Invalid SMILES string provided."

    # Reshape array for a single sample prediction
    fp_reshaped = fp.reshape(1, -1)
    predicted_score = model.predict(fp_reshaped)[0]
    return predicted_score


# Example Prediction (Using Aspirin as a test case)
sample_smiles = "CC(=O)OC1=CC=CC=C1C(=O)O"
predicted_val = predict_new_molecule(sample_smiles)
print(
    f"\nPredicted {target_endpoint} probability for sample structure: {predicted_val:.4f}"
)

Generating chemical fingerprints from SMILES strings...


[03:40:40] DEPRECATION WARNING: please use MorganGenerator
[03:40:40] DEPRECATION WARNING: please use MorganGenerator
[03:40:40] DEPRECATION WARNING: please use MorganGenerator
[03:40:40] DEPRECATION WARNING: please use MorganGenerator
[03:40:40] DEPRECATION WARNING: please use MorganGenerator
[03:40:40] DEPRECATION WARNING: please use MorganGenerator
[03:40:40] DEPRECATION WARNING: please use MorganGenerator
[03:40:40] DEPRECATION WARNING: please use MorganGenerator
[03:40:40] DEPRECATION WARNING: please use MorganGenerator
[03:40:40] DEPRECATION WARNING: please use MorganGenerator
[03:40:40] DEPRECATION WARNING: please use MorganGenerator
[03:40:40] DEPRECATION WARNING: please use MorganGenerator
[03:40:40] DEPRECATION WARNING: please use MorganGenerator
[03:40:40] DEPRECATION WARNING: please use MorganGenerator
[03:40:40] DEPRECATION WARNING: please use MorganGenerator
[03:40:40] DEPRECATION WARNING: please use MorganGenerator
[03:40:40] DEPRECATION WARNING: please use MorganGenerat

Dataset successfully featurized. Matrix shape: (2856, 2048)
Training set size: 2284 molecules
Testing set size: 572 molecules
Training Random Forest model to predict hERG...

=== MODEL PERFORMANCE METRICS ===
Mean Squared Error (MSE): 0.0029
Coefficient of Determination (R² Score): 0.9433

Predicted hERG probability for sample structure: 0.4493


[03:40:43] DEPRECATION WARNING: please use MorganGenerator


In [37]:
import urllib.request

# Correct, updated URL for the PaDEL-Descriptor jar file
url = "https://github.com/yapcw/padeldescriptor/raw/master/PaDEL-Descriptor/PaDEL-Descriptor.jar"

print("Downloading PaDEL-Descriptor.jar...")
try:
    urllib.request.urlretrieve(url, "PaDEL-Descriptor.jar")
    print("PaDEL-Descriptor.jar downloaded successfully!")
except Exception as e:
    print(f"Download failed: {e}")

Download failed: HTTP Error 404: Not Found


In [35]:
df.columns = df.columns.str.strip()

# Target prediction endpoint (e.g., cardiotoxicity)
target_endpoint = "hERG"
df = df.dropna(subset=["SMILES", target_endpoint])

print(f"Loaded {df.shape[0]} molecules for processing.")


# ==========================================
# 2. GENERATE PaDEL DESCRIPTORS
# ==========================================
# PaDEL processes batch jobs efficiently via an external file
temp_smi_file = "temp_molecules.smi"
temp_xml_output = "padel_descriptors.csv"

# Write SMILES strings and a tracking index to a temporary file
df[["SMILES", "cid"]].to_csv(temp_smi_file, sep="\t", index=False, header=False)

print("Running PaDEL engine (extracting 1D & 2D descriptors)...")
try:
    padeldescriptor(
        mol_dir=temp_smi_file,
        d_file=temp_xml_output,
        descriptortypes=None,  # None defaults to all standard 1D & 2D descriptors
        detectaromaticity=True,
        standardizenitro=True,
        standardizetautomers=True,
        threads=2,  # Adjust based on your CPU cores
        removesalt=True,
        log=False,
    )
    print("PaDEL descriptors calculated successfully.")
except Exception as e:
    print(f"Error running PaDEL: {e}")
    # Clean up files before exiting if an error occurs
    if os.path.exists(temp_smi_file):
        os.remove(temp_smi_file)
    raise


# ==========================================
# 3. MATRIX DATA ALIGNMENT & CLEANING
# ==========================================
# Load the freshly calculated features
X_descriptors = pd.read_csv(temp_xml_output)

# PaDEL outputs the molecule identifier as the 'Name' column
# We drop this string column to isolate purely numerical data
X_clean = X_descriptors.drop(columns=["Name"])

# Crucial Step: PaDEL can generate NaN strings or constant columns for some entries.
# 1. Convert any accidental string values to numeric properties, turning errors into NaNs
X_clean = X_clean.apply(pd.to_numeric, errors="coerce")

# 2. Drop columns that contain entirely missing values or zero variance
X_clean = X_clean.dropna(axis=1, how="all")
X_clean = X_clean.loc[:, X_clean.var() > 0.0]

# 3. Fill any isolated internal NaNs with the column average value (Imputation)
X_clean = X_clean.fillna(X_clean.mean())

# Convert to standard structural NumPy array formats
X = X_clean.values
y = df[target_endpoint].values

print(f"Feature processing complete. Final matrix shape: {X.shape}")

# Clean up local temporary generation files
if os.path.exists(temp_smi_file):
    os.remove(temp_smi_file)
if os.path.exists(temp_xml_output):
    os.remove(temp_xml_output)


# ==========================================
# 4. TRAINING AND VALIDATION
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

# Train the Random Forest
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Evaluate performance
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n=== MODEL PERFORMANCE METRICS (PaDEL) ===")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Coefficient of Determination (R² Score): {r2:.4f}")


# ==========================================
# 5. BONUS: FEATURE IMPORTANCE ANALYSIS
# ==========================================
# Because PaDEL columns have explicit physical names, we can see exactly what matters
importances = model.feature_importances_
feature_names = X_clean.columns
indices = np.argsort(importances)[::-1]

print("\n=== TOP 5 MOST IMPORTANT PADEL DESCRIPTORS ===")
for i in range(5):
    print(
        f"{i+1}. Descriptor: {feature_names[indices[i]]} (Relative Weight: {importances[indices[i]]:.4f})"
    )

Loaded 2856 molecules for processing.
Running PaDEL engine (extracting 1D & 2D descriptors)...
PaDEL descriptors calculated successfully.
Feature processing complete. Final matrix shape: (2856, 0)
Training set size: 2284 samples
Testing set size: 572 samples


ValueError: Found array with 0 feature(s) (shape=(2284, 0)) while a minimum of 1 is required by RandomForestRegressor.

In [22]:
df.isnull().sum()

COUNT             0
cid               0
SMILES            0
MW_pub            0
ck pub            0
VERIFY            0
Unnamed: 6     2856
ck_swis         856
MW_swis           0
ESOL Class        0
Unnamed: 10    2856
Unnamed: 11    2856
Unnamed: 12    2856
CK_AD             0
MW_AD             0
caco2             0
hia               0
BBB               0
hERG              0
hERG-10um         0
Ames              0
H-HT              0
Fingerprint    2856
dtype: int64

In [18]:
# # Drop any rows where the SMILES string failed to convert properly
df = df.dropna(subset=["Fingerprint"])
df

,COUNT,cid,SMILES,MW_pub,ck pub,VERIFY,Unnamed: 6,ck_swis,MW_swis,ESOL Class,...,CK_AD,MW_AD,caco2,hia,BBB,hERG,hERG-10um,Ames,H-HT,Fingerprint


In [ ]:

# Construct the feature matrix X and target vector y
X = np.array(df["Fingerprint"].tolist())
y = df[target_endpoint].values

print(f"Dataset successfully featurized. Matrix shape: {X.shape}")


In [ ]:

# ==========================================
# 3. DATA SPLITTING & MACHINE LEARNING
# ==========================================
# Split into 80% training data and 20% test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]} molecules")
print(f"Testing set size: {X_test.shape[0]} molecules")

# Initialize and train the Random Forest Regressor
print(f"Training Random Forest model to predict {target_endpoint}...")
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)


# ==========================================
# 4. MODEL EVALUATION
# ==========================================
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n=== MODEL PERFORMANCE METRICS ===")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Coefficient of Determination (R² Score): {r2:.4f}")


# ==========================================
# 5. PREDICTING ON NEW, UNKNOWN MOLECULES
# ==========================================
def predict_new_molecule(new_smiles_string):
    """Predicts the toxicity score for a brand-new molecule string."""
    fp = smiles_to_padel_fp(new_smiles_string)
    if fp is None:
        return "Invalid SMILES string provided."

    # Reshape array for a single sample prediction
    fp_reshaped = fp.reshape(1, -1)
    predicted_score = model.predict(fp_reshaped)[0]
    return predicted_score


# Example Prediction (Using Aspirin as a test case)
sample_smiles = "CC(=O)OC1=CC=CC=C1C(=O)O"
predicted_val = predict_new_molecule(sample_smiles)
print(
    f"\nPredicted {target_endpoint} value for sample structure: {predicted_val:.4f}"
)